# HHVietSub — OmniVoice GPU Server
Chọn **Runtime → Change runtime type → T4 GPU**, sau đó **Run all**.

In [ ]:
!nvidia-smi
!git clone --depth 1 https://github.com/REPLACE_OWNER/HHVietSub-Colab.git /content/HHVietSub-Colab
!git clone --depth 1 https://github.com/k2-fsa/OmniVoice.git /content/OmniVoice
!pip -q install -e /content/OmniVoice
!pip -q install -r /content/HHVietSub-Colab/requirements.txt
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared

In [ ]:
import os, secrets, subprocess, time, re, threading
token = secrets.token_urlsafe(24)
env = {**os.environ, 'HHVIETSUB_TOKEN': token}
api = subprocess.Popen(['uvicorn','server:app','--host','127.0.0.1','--port','3920'], cwd='/content/HHVietSub-Colab', env=env)
time.sleep(4)
tunnel = subprocess.Popen(['/content/cloudflared','tunnel','--url','http://127.0.0.1:3920'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print('TOKEN BẢO MẬT:', token)
for line in tunnel.stdout:
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        print('URL HHVIETSUB:', match.group(0))
        print('Dán cả URL và TOKEN vào HHVietSub.')
        break
threading.Thread(target=lambda: [print(x.rstrip()) for x in tunnel.stdout], daemon=True).start()